In [33]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date, lower, trim
from pyspark.sql import functions as F
import os
from pyspark.sql import Window

In [34]:
spark = SparkSession.builder \
    .appName("VenchiDataLoading") \
    .getOrCreate()

## Read Data

In [35]:
current_path = os.getcwd()
print(current_path)

parent_path = os.path.dirname(current_path)
print(parent_path)

/Users/matteorossi/Desktop/venchi_project/venchi_jMLE_candidate_pack/submission
/Users/matteorossi/Desktop/venchi_project/venchi_jMLE_candidate_pack


In [36]:
# 2. Construct absolute paths using os.path.join 
# This automatically handles slashes correctly for Mac/Linux/Windows
accounts_path = os.path.join(parent_path, 'data', 'accounts.parquet')
interactions_path = os.path.join(parent_path, 'data', 'interactions.parquet')
products_path = os.path.join(parent_path, 'data', 'products.txt')
sales_path = os.path.join(parent_path, 'data', 'sales.parquet')

# 3. Read the data in Spark
# Spark will now receive the full, absolute path (e.g., /Users/Name/Project/data/accounts.parquet)
accounts_df = spark.read.parquet(accounts_path)
accounts_df.printSchema()
interactions_df = spark.read.parquet(interactions_path)
interactions_df.printSchema()
sales_df = spark.read.parquet(sales_path)
sales_df.printSchema()
products_df = spark.read.csv(products_path, sep='\t', header=True, inferSchema=True)
products_df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- hct: long (nullable = true)
 |-- staff: long (nullable = true)
 |-- turnover_m_usd: long (nullable = true)
 |-- brand_loyalty: long (nullable = true)
 |-- timestamp: string (nullable = true)

root
 |-- interaction_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- duration_mins: long (nullable = true)
 |-- response: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- date: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



### Data Cleaning

In [37]:
accounts_clean = accounts_df.filter(col("account_id").isNotNull())

interactions_clean = interactions_df.filter(col("account_id").isNotNull()) \
    .withColumn("topic_clean", lower(trim(col("topic")))) \
    .withColumn("channel_clean", lower(trim(col("channel")))) \
    .withColumn("response_clean", lower(trim(col("response"))))

sales_clean = sales_df.filter(col("account_id").isNotNull())
products_clean = products_df.filter(col("product_id").isNotNull())

In [38]:
accounts_clean.filter(col("account_id") == "fd242bcbfb732223d528a501ccfcbc7d").show()

+--------------------+---+-----+--------------+-------------+-------------------+
|          account_id|hct|staff|turnover_m_usd|brand_loyalty|          timestamp|
+--------------------+---+-----+--------------+-------------+-------------------+
|fd242bcbfb732223d...| 83|   13|            83|            7|2020-06-07 11:40:37|
|fd242bcbfb732223d...| 91|   18|            87|           14|2020-04-14 03:22:11|
|fd242bcbfb732223d...| 88|   21|            92|           16|2020-01-31 18:22:15|
+--------------------+---+-----+--------------+-------------+-------------------+



In [39]:
# Order by timestamp DESCENDING (newest first)
latest_window = Window.partitionBy("account_id").orderBy(F.col("timestamp").desc())

# Keep only the newest record per account
deduplicated_accounts = (
    accounts_clean
    .withColumn("row_num", F.row_number().over(latest_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num") # Drop the helper column when done
)

In [40]:
accounts_clean.groupBy("account_id").count().orderBy('count').filter(col("count") > 1).toPandas()

,account_id,count
0,d1022401b7eec8871ae15060675defd8,2
1,0ffe7466d9271fdea80561895e646212,2
2,edba0822938758da14ea7bf2115f33a5,2
3,c8937bf01102a4c633d99c2789776ab2,2
4,fb8a2c3a0a50442354fe11ae6b9f9d21,2
...,...,...
2131,fd242bcbfb732223d528a501ccfcbc7d,3
2132,01459f98eb6004b1acec895aad76ed64,3
2133,04c6f523dbbf07accf50428134daf4d5,3
2134,0368996b1a19e1aa1034363420a95fe5,3


In [41]:
deduplicated_accounts.groupBy("account_id").count().orderBy('count').filter(col("count") > 1).toPandas()

,account_id,count


In [27]:
accounts_clean = deduplicated_accounts

In [30]:
sales_clean.groupBy("sale_id").count().orderBy('count').filter(col("count") > 1).toPandas()

,sale_id,count


In [43]:
interactions_clean.groupBy("interaction_id").count().orderBy('count').filter(col("count") > 1).toPandas()

,interaction_id,count


In [45]:
products_clean.groupBy("product_id").count().orderBy('count').filter(col("count") > 1).toPandas()

,product_id,count


## Task 1

### Data Cleaning

,sale_id,count
0,92200368efc0e3d99ae80b950c42c88f,1
1,1c886d3b76e21803bac569c71cb2e2c4,1
2,95681e4c5f8664f6bf7a95feba6a2977,1
3,2d9a6c2713ae72b83ae36d3dba76f1da,1
4,e2b9888058f0539240f2c00b61de4035,1
...,...,...
199995,7d212555f94307bcce1a2c49acd21e16,1
199996,0a01980916f684666d7bfb5d9e38c96e,1
199997,945d2f384826874784aa1e22c6d52731,1
199998,58c72f2d5f36d07807792a8246ed21ef,1


### KPI 1: Total Revenue Generated

In [46]:
sales_with_price = sales_clean.join(products_clean, on="product_id", how="left")
sales_with_price.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



In [47]:
# Aggregate revenue per customer
kpi_revenue = sales_with_price.groupBy("account_id").agg(
    F.sum("price").alias("total_revenue")
)

In [48]:
kpi_revenue.show(10)

+--------------------+-------------+
|          account_id|total_revenue|
+--------------------+-------------+
|e1672d506568deedf...|       121980|
|cb04cb651869ca904...|        56680|
|a054955a13c04e9aa...|       106884|
|115d1256179d8ca02...|        89245|
|ccaf873af5b6fdd3b...|        93540|
|2aca7152120b6433f...|        39077|
|caf1c34f21405ceb6...|        68992|
|79f393197ee0b3251...|        45490|
|0ff82ad074e663021...|        63555|
|21fc105190774346e...|        63909|
+--------------------+-------------+
only showing top 10 rows


In [9]:
## Extra interesting KPI 

kpi_revenue_product = sales_with_price.groupBy("account_id", "product_id").agg(
    F.sum("price").alias("total_revenue")
)


kpi_revenue_category = sales_with_price.groupBy("account_id", "category").agg(
    F.sum("price").alias("total_revenue")
)

### KPI 2: Number of sales interactions in the last 6 months

In [49]:
max_date_row = interactions_clean.select(F.max("date").alias("latest_date")).collect()[0]
latest_date = max_date_row["latest_date"]
print(f"Latest interaction date: {latest_date}")

Latest interaction date: 2021-02-05T00:00:00.000Z


In [50]:
sales_interactions_6m = interactions_clean.filter(
    (F.col("date") >= F.date_sub(F.lit(latest_date), 180))
)
sales_interactions_6m.show(10)

+--------------------+------------+-------------+--------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------+--------------+
|      interaction_id|     channel|duration_mins|response|              topic|                date|          account_id|          product_id|        topic_clean|channel_clean|response_clean|
+--------------------+------------+-------------+--------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------+--------------+
|9e8a83833a5a5dafb...|       email|            6|positive|against competition|2020-10-17T00:00:...|544feaa1526ef94f1...|2a38ce238f4b3a52b...|against competition|        email|      positive|
|17f65a62b55523036...|face to face|           82|   mixed|  available finance|2021-01-24T00:00:...|f63ebf8ff4a382c2e...|0eed3f6963570b036...|  available finance| face to face|         mixed|
|909f3a6eca8fbf6b7...|face to face|          

In [51]:
# Aggregate count per customer
kpi_recent_sales_interactions = sales_interactions_6m.groupBy("account_id").agg(
    F.count("interaction_id").alias("sales_interactions_last_6m")
)

In [52]:
kpi_recent_sales_interactions.filter(col("sales_interactions_last_6m") > 2).show(10)

+--------------------+--------------------------+
|          account_id|sales_interactions_last_6m|
+--------------------+--------------------------+
|b9cd1be320b532234...|                         3|
+--------------------+--------------------------+



### KPI 3 Distinct Category

In [53]:
kpi_distinct_cateogory = sales_with_price.groupBy("account_id").agg(
    F.countDistinct("category").alias("distinct_categories") 
)

In [54]:
kpi_distinct_cateogory.show(10)

+--------------------+-------------------+
|          account_id|distinct_categories|
+--------------------+-------------------+
|97830d3b951ec86b8...|                  4|
|936ce27e14558596f...|                  2|
|59d117fcbc6b370f7...|                  4|
|e5131a914d7c3b9f7...|                  4|
|54a774e3b8b30834d...|                  4|
|149c86cc1d278a2d5...|                  4|
|98f62e8b624d99682...|                  4|
|a054955a13c04e9aa...|                  4|
|4bf028c37e2c27dfe...|                  4|
|edba0822938758da1...|                  4|
+--------------------+-------------------+
only showing top 10 rows


### KPI: Average interaction for client

In [55]:
kpi_avg_duration = interactions_clean.groupBy("account_id").agg(
    F.round(F.avg("duration_mins"), 2).alias("avg_interaction_duration_mins")
)

In [56]:
kpi_avg_duration.show(5)

+--------------------+-----------------------------+
|          account_id|avg_interaction_duration_mins|
+--------------------+-----------------------------+
|285fffa8d29a9f9b2...|                         14.0|
|26b8c340d56a7e3bb...|                        102.5|
|ae9e95dc1fb78abec...|                         70.5|
|98f62e8b624d99682...|                        121.0|
|fdf8a784a1d444a52...|                       132.67|
+--------------------+-----------------------------+
only showing top 5 rows


## TASK 2

### Putting Data Together

In [57]:
analytical_dataset = accounts_clean \
    .join(kpi_revenue, on="account_id", how="left") \
    .join(kpi_recent_sales_interactions, on="account_id", how="left") \
    .join(kpi_distinct_cateogory, on="account_id", how="left") \
    .join(kpi_avg_duration, on="account_id", how="left")

In [58]:
shape = (analytical_dataset.count(), len(analytical_dataset.columns))
print(shape)

(12205, 10)


In [59]:
analytical_dataset = analytical_dataset.fillna({
    "total_revenue": 0,
    "sales_interactions_last_6m": 0,
    "distinct_categories": 0,
    "avg_interaction_duration_mins": 0
})

In [62]:
analytical_dataset.toPandas()

,account_id,hct,staff,turnover_m_usd,brand_loyalty,timestamp,total_revenue,sales_interactions_last_6m,distinct_categories,avg_interaction_duration_mins
0,d1022401b7eec8871ae15060675defd8,3,5,22,4,2020-06-07 11:40:37,60872,0,4,3.00
1,0ffe7466d9271fdea80561895e646212,42,15,3,8,2020-06-07 11:40:37,28287,0,3,0.00
2,edba0822938758da14ea7bf2115f33a5,50,16,46,6,2020-06-07 11:40:37,47898,0,4,59.00
3,c8937bf01102a4c633d99c2789776ab2,65,20,1,8,2020-06-07 11:40:37,32737,0,3,166.00
4,cb04cb651869ca9045ab52aefcfd4c8c,12,7,14,10,2020-06-07 11:40:37,56680,0,4,0.00
...,...,...,...,...,...,...,...,...,...,...
12200,de194e239a3dc5e4144aaf3c1663bf89,54,24,25,19,2020-01-31 18:22:15,95951,0,4,0.00
12201,369534733003f9af8ae23ed49a26e4ec,6,11,44,12,2020-01-31 18:22:15,52329,0,4,178.00
12202,27645453873beab099d186af292564c7,82,9,78,13,2020-01-31 18:22:15,83597,0,4,80.67
12203,0182d13afa00f0d4aa11060fae51e74d,58,10,17,10,2020-01-31 18:22:15,21822,1,4,120.50


### Developing the model

In [63]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [64]:

# ==========================================
# 1. PREPARE DATA FOR MACHINE LEARNING
# ==========================================
# Ensure no nulls exist in the original account columns before vectorization.
ml_data = analytical_dataset.fillna(0)

# Define the features we want our model to learn from.
# We mix firmographics (turnover) with our engineered behavioral KPIs.
feature_cols = [
    "turnover_m_usd", 
    "brand_loyalty", 
    "total_revenue", 
    "sales_interactions_last_6m", 
    "distinct_categories", 
    "avg_interaction_duration_mins"
]

# VectorAssembler combines our individual feature columns into a single vector column called 'features'
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled_data = assembler.transform(ml_data)

# StandardScaler normalizes the features so they have a mean of 0 and std dev of 1.
# This prevents large numerical values (like revenue) from dominating the K-Means distance calculations.
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_data)
scaled_data = scaler_model.transform(assembled_data)


# ==========================================
# 2. DETERMINE THE NUMBER OF CLUSTERS (k)
# ==========================================
# We iterate through potential cluster sizes (2 to 5) and calculate the Silhouette Score.
# A score closer to 1 indicates well-separated, dense clusters.

evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="scaled_features", metricName="silhouette")
best_k = 2
best_score = -1

print("Evaluating cluster sizes...")
for k in range(2, 6):
    kmeans = KMeans(featuresCol="scaled_features", k=k, seed=42)
    model = kmeans.fit(scaled_data)
    predictions = model.transform(scaled_data)
    score = evaluator.evaluate(predictions)
    print(f"Silhouette Score for k={k}: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"\nSelected optimal number of clusters: {best_k} (based on highest Silhouette Score)")


# ==========================================
# 3. TRAIN FINAL MODEL & ASSIGN SEGMENTS
# ==========================================
# Train the final model using the best K
final_kmeans = KMeans(featuresCol="scaled_features", k=best_k, seed=42)
final_model = final_kmeans.fit(scaled_data)

# Apply the model to our dataset to generate the 'prediction' column (the segment ID)
enriched_data = final_model.transform(scaled_data)

# Rename 'prediction' to 'customer_segment' for business clarity
enriched_data = enriched_data.withColumnRenamed("prediction", "customer_segment")


# ==========================================
# 4. PROFILE THE SEGMENTS
# ==========================================
# Calculate the average values for our KPIs per segment to understand who these customers are.
segment_profiles = enriched_data.groupBy("customer_segment").agg(
    F.count("account_id").alias("customer_count"),
    F.round(F.avg("turnover_m_usd"), 2).alias("avg_turnover"),
    F.round(F.avg("brand_loyalty"), 2).alias("avg_loyalty"),
    F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
    F.round(F.avg("sales_interactions_last_6m"), 2).alias("avg_recent_interactions"),
    F.round(F.avg("distinct_categories"), 2).alias("avg_distinct_categories")
).orderBy("customer_segment")

print("\n--- Segment Profiles ---")
segment_profiles.show()


Evaluating cluster sizes...


26/05/10 19:01:36 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Silhouette Score for k=2: 0.3671
Silhouette Score for k=3: 0.3008
Silhouette Score for k=4: 0.3458
Silhouette Score for k=5: 0.4230

Selected optimal number of clusters: 5 (based on highest Silhouette Score)

--- Segment Profiles ---
+----------------+--------------+------------+-----------+-----------+-----------------------+-----------------------+
|customer_segment|customer_count|avg_turnover|avg_loyalty|avg_revenue|avg_recent_interactions|avg_distinct_categories|
+----------------+--------------+------------+-----------+-----------+-----------------------+-----------------------+
|               0|          3910|       32.15|       6.87|   47127.48|                    0.0|                   3.99|
|               1|          1394|       17.56|       3.83|   18805.49|                   0.02|                   2.52|
|               2|           803|       60.91|       5.86|   68299.11|                   1.03|                   3.92|
|               3|          2921|       62.47|      

In [65]:
# We drop the ML-specific vector columns before saving to keep the final dataset clean.
final_export_df = enriched_data.drop("raw_features", "scaled_features")

output_path = os.path.join(parent_path, 'output', 'enriched_customer_segments.parquet')

# Write to Parquet (overwrite if it already exists)
final_export_df.write.mode("overwrite").parquet(output_path)
print(f"\nEnriched dataset saved to: {output_path}")


Enriched dataset saved to: /Users/matteorossi/Desktop/venchi_project/venchi_jMLE_candidate_pack/output/enriched_customer_segments.parquet


### Idea of Keeping only the recent Interactions

In [ ]:
analytical_dataset = analytical_dataset.filter(col("sales_interactions_last_6m").isNotNull())

In [ ]:
shape = (analytical_dataset.count(), len(analytical_dataset.columns))
print(shape)

In [ ]:
# Order by total_revenue from highest to lowest
sorted_df = analytical_dataset.orderBy(F.col("sales_interactions_last_6m").desc())

sorted_df.show(5)